# QAI Models Setup Notebook

This notebook sets up quantum-AI (QAI) models for development and deployment. It covers environment configuration, model initialization, and verification testing.

**Key Steps:**
1. Import required libraries and check dependencies
2. Install/verify QAI framework packages
3. Initialize quantum environment and backends
4. Load and configure pre-trained models
5. Run verification tests

**Workspace:** `quantum-ai/`
**Framework:** PennyLane + Qiskit
**Backends:** lightning.qubit, default.qubit, qasm_simulator

## 1. Import Required Libraries

In [ ]:
import sys
import os
from pathlib import Path
import json
from datetime import datetime

# Add quantum-ai src to path
QAI_ROOT = Path.cwd()
if "quantum-ai" not in str(QAI_ROOT):
    QAI_ROOT = Path("/workspaces/AI/quantum-ai") if Path("/workspaces/AI/quantum-ai").exists() else Path(".")

sys.path.insert(0, str(QAI_ROOT / "src"))

print(f"✅ Workspace: {QAI_ROOT}")
print(f"✅ Python version: {sys.version}")
print(f"✅ Path configured")

# Import data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"✅ NumPy: {np.__version__}")
print(f"✅ Pandas: {pd.__version__}")
print(f"✅ Matplotlib: {matplotlib.__version__}")

In [ ]:
import matplotlib
print(f"✅ Matplotlib: {matplotlib.__version__}")

# Test importing core quantum libraries
try:
    import qiskit
    print(f"✅ Qiskit: {qiskit.__version__}")
except ImportError as e:
    print(f"⚠️  Qiskit not installed: {e}")

try:
    import pennylane as qml
    print(f"✅ PennyLane: {qml.__version__}")
except ImportError as e:
    print(f"⚠️  PennyLane not installed: {e}")

try:
    import torch
    print(f"✅ PyTorch: {torch.__version__}")
except ImportError as e:
    print(f"⚠️  PyTorch not installed: {e}")

try:
    from sklearn import datasets
    print(f"✅ scikit-learn imported")
except ImportError as e:
    print(f"⚠️  scikit-learn not installed: {e}")

print("\n✅ All imports completed")

## 2. Install QAI Dependencies

In [ ]:
import subprocess

# Check if we need to install dependencies
required_packages = {
    "qiskit": "Qiskit",
    "pennylane": "PennyLane",
    "torch": "PyTorch",
    "sklearn": "scikit-learn",
    "azure": "Azure SDK",
    "pyyaml": "PyYAML"
}

missing_packages = []

print("📦 Checking dependencies...\n")
for package, name in required_packages.items():
    try:
        __import__(package)
        print(f"✅ {name:20s} - Installed")
    except ImportError:
        print(f"❌ {name:20s} - MISSING")
        missing_packages.append(package)

if missing_packages:
    print(f"\n⚠️  Installing missing packages: {', '.join(missing_packages)}")
    for package in missing_packages:
        print(f"\n📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    print("\n✅ All dependencies installed")
else:
    print("\n✅ All dependencies already installed")

## 3. Initialize QAI Environment

In [ ]:
import yaml
import torch

print("🔧 Initializing QAI Environment\n")

# 1. Configure PyTorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📊 Device: {device}")
if torch.cuda.is_available():
    print(f"   • GPU: {torch.cuda.get_device_name(0)}")
    print(f"   • CUDA Version: {torch.version.cuda}")
    print(f"   • Devices: {torch.cuda.device_count()}")

# 2. Load quantum configuration
config_file = QAI_ROOT / "config" / "quantum_config.yaml"
if config_file.exists():
    with open(config_file, 'r') as f:
        config = yaml.safe_load(f)
    print(f"\n⚙️  Configuration loaded: {config_file.name}")
    print(f"   • ML Model Qubits: {config['ml']['model']['n_qubits']}")
    print(f"   • ML Model Layers: {config['ml']['model']['n_layers']}")
    print(f"   • Quantum Backend: {config['quantum']['simulator']['backend']}")
    print(f"   • Simulator Shots: {config['quantum']['simulator']['shots']}")
else:
    print(f"⚠️  Config file not found: {config_file}")
    config = {}

# 3. Import quantum frameworks
import pennylane as qml
from pennylane import numpy as pnp
import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

print(f"\n✅ PennyLane: {qml.__version__}")
print(f"✅ Qiskit: {qiskit.__version__}")

# 4. Set up available backends
print(f"\n📦 Available PennyLane Backends:")
available_backends = [
    "lightning.qubit",
    "default.qubit",
    "default.qubit.legacy",
    "default.mixed"
]

for backend in available_backends:
    try:
        dev = qml.device(backend, wires=2)
        print(f"   ✅ {backend}")
    except Exception as e:
        print(f"   ⚠️  {backend}: {str(e)[:50]}")

print(f"\n✅ QAI Environment initialized")

## 4. Create Model Registry and Checkpoint Structure

In [ ]:
print("📁 Setting up checkpoint structure...\n")

# Create checkpoint directories
checkpoint_dirs = [
    "checkpoints",
    "checkpoints/quantum_classifier",
    "checkpoints/variational_circuits",
    "checkpoints/grover_algorithms",
    "checkpoints/ensemble_models",
    "checkpoints/best_models",
    "checkpoints/experiments",
    "checkpoints/backups",
]

for dir_path in checkpoint_dirs:
    full_path = QAI_ROOT / dir_path
    full_path.mkdir(parents=True, exist_ok=True)

print(f"✅ Created {len(checkpoint_dirs)} checkpoint directories")

# Create model registry
registry = {
    "created_at": datetime.now().isoformat(),
    "version": "1.0",
    "models": {
        "quantum_classifier": {
            "type": "hybrid_quantum_classical",
            "framework": "pennylane",
            "backend": config.get("quantum", {}).get("simulator", {}).get("backend", "lightning.qubit"),
            "status": "ready",
            "config": {
                "n_qubits": config.get("ml", {}).get("model", {}).get("n_qubits", 4),
                "n_layers": config.get("ml", {}).get("model", {}).get("n_layers", 2)
            },
            "checkpoints": []
        },
        "variational_circuit": {
            "type": "parametrized_quantum_circuit",
            "framework": "pennylane",
            "backend": config.get("quantum", {}).get("simulator", {}).get("backend", "default.qubit"),
            "status": "ready",
            "config": {
                "n_qubits": config.get("ml", {}).get("model", {}).get("n_qubits", 4),
                "n_layers": 3
            },
            "checkpoints": []
        },
        "grover_circuit": {
            "type": "quantum_algorithm",
            "framework": "qiskit",
            "backend": "qasm_simulator",
            "status": "ready",
            "config": {"n_qubits": 3, "shots": 1000},
            "checkpoints": []
        }
    },
    "datasets": {
        "moons": {"status": "available", "features": 2, "samples": 300},
        "iris": {"status": "available", "features": 4, "samples": 150},
        "banknote": {"status": "available", "features": 4, "samples": 1372}
    }
}

registry_file = QAI_ROOT / "checkpoints" / "registry.json"
with open(registry_file, 'w') as f:
    json.dump(registry, f, indent=2)

print(f"✅ Model registry created: {registry_file.name}\n")
print("📋 Registered Models:")
for model_name, model_info in registry["models"].items():
    print(f"   • {model_name:30s} ({model_info['framework']})")

print(f"\n📊 Available Datasets:")
for dataset_name, dataset_info in registry["datasets"].items():
    print(f"   • {dataset_name:30s} ({dataset_info['samples']} samples, {dataset_info['features']} features)")

## 5. Configure Model Parameters

In [ ]:
print("⚙️  Configuring Model Parameters\n")

# Training configuration templates
training_configs = {
    "default": {
        "model": "quantum_classifier",
        "dataset": "moons",
        "epochs": 100,
        "learning_rate": 0.01,
        "batch_size": 8,
        "backend": "lightning.qubit",
        "optimizer": "adam"
    },
    "intensive": {
        "model": "variational_circuit",
        "dataset": "iris",
        "epochs": 200,
        "learning_rate": 0.001,
        "batch_size": 4,
        "backend": "default.qubit",
        "optimizer": "sgd"
    },
    "quick": {
        "model": "quantum_classifier",
        "dataset": "moons",
        "epochs": 10,
        "learning_rate": 0.01,
        "batch_size": 16,
        "backend": "lightning.qubit",
        "optimizer": "adam"
    }
}

# Performance targets
performance_targets = {
    "quantum_classifier_accuracy": 0.85,
    "variational_circuit_loss": 0.05,
    "training_time_seconds": 300,
    "inference_time_ms": 100
}

# Create training config file
training_config_file = QAI_ROOT / "checkpoints" / "training_config.json"
with open(training_config_file, 'w') as f:
    json.dump({
        "training_sessions": training_configs,
        "performance_targets": performance_targets
    }, f, indent=2)

print("📋 Training Configurations:")
for config_name, config_params in training_configs.items():
    print(f"\n   {config_name.upper()}:")
    print(f"      • Model: {config_params['model']}")
    print(f"      • Dataset: {config_params['dataset']}")
    print(f"      • Epochs: {config_params['epochs']}")
    print(f"      • Learning Rate: {config_params['learning_rate']}")

print(f"\n🎯 Performance Targets:")
for target, value in performance_targets.items():
    if isinstance(value, float) and value < 1:
        print(f"   • {target:40s} ≤ {value}")
    else:
        print(f"   • {target:40s} ≤ {value:.0f}")

print(f"\n✅ Configuration saved: {training_config_file.name}")

## 6. Initialize Sample Quantum Circuit

In [ ]:
print("🔬 Initializing Sample Quantum Circuit\n")

# Create a simple quantum circuit using PennyLane
n_qubits = 2
dev = qml.device("lightning.qubit", wires=n_qubits)

@qml.qnode(dev)
def simple_circuit(params):
    """Simple parameterized quantum circuit"""
    for i in range(n_qubits):
        qml.RX(params[i], wires=i)
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])
    return qml.expval(qml.PauliZ(0))

# Test the circuit
params = np.array([0.1, 0.2])
result = simple_circuit(params)

print(f"✅ Sample PennyLane Circuit:")
print(f"   • Qubits: {n_qubits}")
print(f"   • Type: Parameterized VQC")
print(f"   • Backend: lightning.qubit")
print(f"   • Result: {result:.4f}")

# Create a Qiskit circuit
print(f"\n✅ Sample Qiskit Circuit:")
qr = QuantumRegister(2, 'q')
cr = ClassicalRegister(2, 'c')
qc = QuantumCircuit(qr, cr)
qc.h(qr[0])
qc.cx(qr[0], qr[1])
qc.measure(qr, cr)
print(f"   • Qubits: 2")
print(f"   • Type: Bell State")
print(f"   • Gates: H, CNOT, Measure")
print(f"\n{qc}")

print(f"\n✅ Quantum circuits initialized successfully")

## 7. Verify Model Setup and Generate Report

In [ ]:
print("✅ VERIFYING QAI MODELS SETUP\n")
print("="*70)

# Create verification report
verification_report = {
    "timestamp": datetime.now().isoformat(),
    "status": "completed",
    "environment": {
        "python_version": sys.version,
        "pytorch_available": torch.cuda.is_available(),
        "device": str(device),
        "workspace": str(QAI_ROOT)
    },
    "frameworks": {
        "pennylane": qml.__version__,
        "qiskit": qiskit.__version__,
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__
    },
    "models_created": list(registry["models"].keys()),
    "datasets_available": list(registry["datasets"].keys()),
    "checkpoint_dirs_created": len(checkpoint_dirs),
    "files_created": {
        "registry": str(registry_file),
        "training_config": str(training_config_file)
    }
}

# Save verification report
report_file = QAI_ROOT / "checkpoints" / "setup_report.json"
with open(report_file, 'w') as f:
    json.dump(verification_report, f, indent=2)

print("\n📋 SETUP VERIFICATION REPORT")
print("="*70)
print(f"\n✅ Environment:")
print(f"   • Python: {sys.version.split()[0]}")
print(f"   • PyTorch: {torch.__version__}")
print(f"   • Device: {device}")

print(f"\n✅ Quantum Frameworks:")
print(f"   • PennyLane: {qml.__version__}")
print(f"   • Qiskit: {qiskit.__version__}")

print(f"\n✅ Models Initialized: {len(registry['models'])}")
for model_name in registry["models"]:
    print(f"   • {model_name}")

print(f"\n✅ Datasets Available: {len(registry['datasets'])}")
for dataset_name in registry["datasets"]:
    print(f"   • {dataset_name}")

print(f"\n✅ Checkpoint Directories: {len(checkpoint_dirs)}")
print(f"✅ Configuration Files: 2 (registry.json, training_config.json)")

print(f"\n📁 Directory Structure:")
print(f"   quantum-ai/")
print(f"   ├── checkpoints/")
print(f"   │   ├── quantum_classifier/")
print(f"   │   ├── variational_circuits/")
print(f"   │   ├── grover_algorithms/")
print(f"   │   ├── ensemble_models/")
print(f"   │   ├── best_models/")
print(f"   │   ├── registry.json")
print(f"   │   ├── training_config.json")
print(f"   │   ├── metrics.json")
print(f"   │   └── setup_report.json")

print(f"\n🚀 Next Steps:")
print(f"   1. Load training data:      from sklearn import datasets")
print(f"   2. Create quantum model:    python examples/train_models.py")
print(f"   3. Run local simulation:    python examples/run_simulations.py")
print(f"   4. Launch dashboard:        ./start_dashboard.sh")
print(f"   5. Submit to Azure:         python azure_quantum_deploy.py")

print(f"\n📊 Report saved: {report_file.name}")
print("="*70)
print(f"\n✅ QAI MODELS SETUP COMPLETE!")

## 8. Load Training Data

In [ ]:
from sklearn.datasets import make_moons, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("📊 Loading Training Datasets\n")

# Load Moons dataset
print("1. Moons Dataset (Binary Classification):")
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)
X_train_m, X_val_m, y_train_m, y_val_m = train_test_split(X_moons, y_moons, test_size=0.2, random_state=42)

# Normalize
scaler_m = StandardScaler()
X_train_m = scaler_m.fit_transform(X_train_m)
X_val_m = scaler_m.transform(X_val_m)

# Pad to 4 dimensions for quantum circuits
X_train_m_padded = np.pad(X_train_m, ((0, 0), (0, 2)), mode='constant')
X_val_m_padded = np.pad(X_val_m, ((0, 0), (0, 2)), mode='constant')

print(f"   ✅ Training samples: {len(X_train_m)}")
print(f"   ✅ Validation samples: {len(X_val_m)}")
print(f"   ✅ Features: 2 → 4 (padded for quantum)")
print(f"   ✅ Classes: 2 (binary)")

# Load Iris dataset
print("\n2. Iris Dataset (Multi-class Classification):")
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_train_i, X_val_i, y_train_i, y_val_i = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

# Normalize
scaler_i = StandardScaler()
X_train_i = scaler_i.fit_transform(X_train_i)
X_val_i = scaler_i.transform(X_val_i)

print(f"   ✅ Training samples: {len(X_train_i)}")
print(f"   ✅ Validation samples: {len(X_val_i)}")
print(f"   ✅ Features: 4")
print(f"   ✅ Classes: 3 (setosa, versicolor, virginica)")

# Store datasets for later use
datasets = {
    "moons": {
        "X_train": X_train_m_padded,
        "X_val": X_val_m_padded,
        "y_train": y_train_m,
        "y_val": y_val_m,
        "scaler": scaler_m
    },
    "iris": {
        "X_train": X_train_i,
        "X_val": X_val_i,
        "y_train": y_train_i,
        "y_val": y_val_i,
        "scaler": scaler_i
    }
}

print(f"\n✅ Datasets loaded: {len(datasets)}")
print(f"✅ Ready for training!")

## 9. Train Quantum Model (Quick Demo)

In [ ]:
print("🔄 QUICK TRAINING DEMO\n")
print("="*70)

# Create a simple quantum classifier
n_qubits = 4
n_layers = 2

# Define a trainable quantum circuit
@qml.qnode(dev)
def variational_circuit(params, x):
    """Parameterized quantum circuit with data encoding"""
    # Data encoding
    for i in range(min(n_qubits, len(x))):
        qml.RY(x[i], wires=i)
    
    # Variational layers
    for layer in range(n_layers):
        for i in range(n_qubits):
            qml.RX(params[layer, i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
    
    return qml.expval(qml.PauliZ(0))

# Initialize parameters
params = pnp.random.randn(n_layers, n_qubits, requires_grad=True)

print("📋 Model Configuration:")
print(f"   • Qubits: {n_qubits}")
print(f"   • Layers: {n_layers}")
print(f"   • Backend: lightning.qubit")
print(f"   • Parameters: {params.size}")

# Training loop - quick demo with 5 iterations
print(f"\n🚀 Training Loop (5 iterations):")
learning_rate = 0.01
opt = qml.GradientDescentOptimizer(stepsize=learning_rate)

# Use first sample from moons dataset
X_sample = X_train_m_padded[:5]
y_sample = y_train_m[:5]

training_losses = []

for epoch in range(5):
    loss = 0
    for i, (x, y) in enumerate(zip(X_sample, y_sample)):
        # Predict
        pred = variational_circuit(params, x)
        # Simple MSE loss
        sample_loss = (pred - (2*y - 1))**2  # Convert to -1, 1
        loss += sample_loss
        
        # Update
        params, loss_val = opt.step(lambda p: (variational_circuit(p, x) - (2*y - 1))**2, params)
    
    avg_loss = loss / len(X_sample)
    training_losses.append(avg_loss)
    print(f"   Epoch {epoch+1:2d} | Loss: {avg_loss:.4f}")

print(f"\n✅ Training Demo Completed!")
print(f"   • Final Loss: {training_losses[-1]:.4f}")
print(f"   • Improvement: {(training_losses[0] - training_losses[-1]):.4f}")

# Plot losses
plt.figure(figsize=(8, 4))
plt.plot(training_losses, marker='o', linewidth=2, markersize=8)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Quantum Model Training - Quick Demo")
plt.grid(True, alpha=0.3)
plt.tight_layout()

demo_plot = QAI_ROOT / "results" / "training_demo.png"
demo_plot.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(demo_plot, dpi=100)
plt.show()

print(f"\n📊 Plot saved: {demo_plot.name}")

## 10. Next Steps & Resources

### Training Scripts Available

#### Quick Start Commands:

**1. Run Full Training Suite:**
```bash
python examples/train_models.py
```
- Trains on Moons, Iris, and Circles datasets
- Generates accuracy plots
- Saves results to `results/`

**2. Start Web Dashboard:**
```bash
# Windows
start_dashboard.bat
# Linux/Mac
./start_dashboard.sh
```
- Access at: http://localhost:5000
- Real-time training visualization
- Interactive parameter tuning
- Live loss/accuracy charts

**3. Run Local Simulations:**
```bash
python examples/run_simulations.py
```
- Test Bell states, superposition, gradients
- Visualize quantum state evolution
- No training required

**4. Test Azure Integration:**
```bash
python examples/azure_integration.py
```
- Verify Azure Quantum configuration
- Test provider connectivity
- Estimate costs

### Model Files & Checkpoints

```
quantum-ai/
├── checkpoints/
│   ├── quantum_classifier/      # Trained hybrid models
│   ├── variational_circuits/    # VQC checkpoints
│   ├── grover_algorithms/       # Search algorithm results
│   ├── best_models/             # Top performers
│   ├── registry.json            # Model registry
│   └── training_config.json     # Training configurations
├── results/
│   ├── training_demo.png        # Generated plots
│   ├── accuracy_comparison.png  # Model comparison
│   └── *.png                    # Visualization results
└── logs/
    └── training_*.log           # Training logs
```

### Performance Tips

**GPU Optimization:**
- Use `backend: lightning.qubit` for fastest simulations
- Enable CUDA if available: `torch.cuda.is_available()`
- Reduce batch size if GPU memory limited

**Training Optimization:**
- Start with small `n_layers` (2-3) then increase
- Use `learning_rate: 0.01` initially
- Increase `epochs` gradually for better convergence

**Azure Quantum:**
- Test locally first with `backend: qiskit_aer`
- Then use `backend: azure_ionq_simulator`
- Set `azure_confirm_cost: true` only for real QPU

### Troubleshooting

| Issue | Solution |
|-------|----------|
| Import errors | `pip install -r requirements.txt` |
| CUDA out of memory | Reduce batch size or use CPU backend |
| Dashboard won't start | Check port 5000 is free: `netstat -ano \| findstr 5000` |
| Training hangs | Reduce dataset size or use `backend: lightning.qubit` |
| Azure connection fails | Run `az login` and verify credentials |

### Documentation

- **[QUICK_REFERENCE.md](QUICK_REFERENCE.md)** - Quick command reference
- **[README.md](README.md)** - Full documentation
- **[QUANTUM_TESTING_SUMMARY.md](QUANTUM_TESTING_SUMMARY.md)** - Test results
- **[MCP_SERVER_README.md](MCP_SERVER_README.md)** - MCP server setup

---

**Ready to train! Run any of the commands above to get started.** 🚀